In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import numpy as np
from matplotlib import pyplot as plt
import psutil

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

In [ ]:
# load features and edges, build PyG data object
feat = torch.load('../data/canwell_features.pt')
x        = feat['x']
y        = feat['y']
is_slope = feat['is_slope']
is_basin = feat['is_basin']
has_diff = feat['has_diff']

edge_index = torch.load('../data/canwell_edgidx.pt')

data = Data(x=x, edge_index=edge_index, y=y)
data.is_slope = is_slope
data.is_basin = is_basin
data.has_diff = has_diff

print(f"nodes: {data.num_nodes:,}")
print(f"edges: {data.num_edges:,}")
print(f"node features: {data.num_node_features}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

In [ ]:
from sklearn.preprocessing import StandardScaler

# normalize node features
x_np = x.numpy()
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_np)
data.x = torch.tensor(x_scaled, dtype=torch.float)
print("data scaled")

In [ ]:
# define masks
# training nodes: has diff value AND is slope, randomly select 70%
slope_with_diff  = (is_slope & has_diff).nonzero(as_tuple=True)[0]
print(f"slope nodes with diff: {len(slope_with_diff):,}")

In [ ]:
# random 70/15/15 on slopediff nodes
perm = torch.randperm(len(slope_with_diff))
n = len(slope_with_diff)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

train_idx = slope_with_diff[perm[:train_end]]
val_idx   = slope_with_diff[perm[train_end:val_end]]
test_idx  = slope_with_diff[perm[val_end:]]

# basin nodes never masked

print(f"train nodes: {len(train_idx):,}")
print(f"val nodes:   {len(val_idx):,}")
print(f"test nodes:  {len(test_idx):,}")
print(f"basin nodes: {is_basin.sum():,}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

In [ ]:
# add masking feature and set up input

# add is_masked as 5th input feature
# train/val/test slope nodes mask=1 -- their diff is hidden from INPUT
# basin nodes get mask=0 -- diff visible as context, added as 6th feature

is_masked = torch.zeros(data.num_nodes, dtype=torch.float)
is_masked[train_idx] = 1.0
is_masked[val_idx] = 1.0
is_masked[test_idx] = 1.0

# diff as input feature: visible for basin nodes, 0 for masked (slope) nodes
diff_input = torch.zeros(data.num_nodes, dtype=torch.float)
diff_input[is_basin & has_diff] = y[is_basin & has_diff]

# normalize diff_input
diff_mean = diff_input[is_basin & has_diff].mean().item()
diff_std  = diff_input[is_basin & has_diff].std().item()
diff_input[is_basin & has_diff] = (diff_input[is_basin & has_diff] - diff_mean)/diff_std

# stack into final feature matrix:
#       [elev, slope, aspect, doubslope, is_masked, diff_input]

data.x = torch.cat([data.x,
                    is_masked.unsqueeze(1),
                    diff_input.unsqueeze(1)], dim=1)

print(f"final feature matrix: {data.x.shape}")
print(f"features: elev,slope,aspect,doubslope,is_masked,diff_input")
print(f"ram: {psutil.virtual_memory().percent}%")

In [ ]:
# define GraphSAGE model:

class CanwellSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super(CanwellSAGE, self).__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)

        # regression head
        self.lin1 = Linear(hidden_channels, hidden_channels//2)
        self.lin2 = Linear(hidden_channels//2, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)

        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)

        x = F.relu(self.conv3(x, edge_index))

        x = F.relu(self.lin1(x))
        x = self.lin2(x)

        return x.squeeze(1)

In [ ]:
model = CanwellSAGE(in_channels=6, hidden_channels=64).to(device)
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# normalize target y for training stability
y_mean = y[train_idx].mean().item()
y_std  = y[train_idx].std().item()
y_norm = (y-y_mean)/y_std
data.y = y_norm

In [ ]:
# set up neighborloader for mini-batch training
train_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10], # sample 10 neighbors per layer
    batch_size=512,
    input_nodes=train_idx,
    shuffle=True,
)

val_loader = NeighborLoader(
    data,
    num_neighbors=[10,10,10],
    batch_size=512,
    input_nodes=val_idx,
    shuffle=False,
)

print(f"train batches: {len(train_loader)}")
print(f"val batches:   {len(val_loader)}")
print(f"y mean: {y_mean:.3f}, std: {y_std:.3f}")
print(f"ram: {psutil.virtual_memory().percent:.1f}")

In [ ]:
# training and validation functions
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

def train():
    model.train()
    total_loss = 0
    count = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        # only compute loss on target nodes (first batch_size nodes)
        out = out[:batch.batch_size]
        target = batch.y[:batch.batch_size]
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        count += batch.batch_size
    return total_loss/count

def validate():
    model.eval()
    total_loss = 0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            out = out[:batch.batch_size]
            target = batch.y[:batch.batch_size]
            loss = criterion(out, target)
            total_loss += loss.item() * batch.batch_size
            count += batch.batch_size
        return total_loss/count

print("train and val functions ready")

In [ ]:
# training loop with early stopping
best_val_loss = float('inf')
best_model = None
patience = 10
no_improve = 0
max_epochs = 100
train_losses = []
val_losses = []

for epoch in range(max_epochs):
    train_loss = train()
    val_loss   = validate()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 5 == 0:
        print(f"epoch: {epoch:03d} | train mse: {train_loss:.4f} | val mse: {val_loss:.4f}")

    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_model = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        print(f" new best: {best_val_loss:.4f} at epoch: {epoch}")
        no_improve = 0
    else:
        no_improve+=1

    if no_improve >= patience:
        print(f"\nearly stopping at epoch {epoch}")
        print(f"best epoch: {best_epoch}, best val mse: {best_val_loss:.4f}")
        model.load_state_dict(best_model)
        break

print("\ntraining complete")

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)